# 🧠 Clasificación de Décadas por Texto — Etapa 2: Deep Learning
## Curso: Aprendizaje de Máquina 2026-10 — Universidad de los Andes

---

### 👥 Información del Grupo

| Campo         | Valor                        |
|---------------|------------------------------|
| **Nombre 1**  | _Completar con nombre_       |
| **Nombre 2**  | _Completar con nombre_       |
| **Nombre 3**  | _Completar con nombre_       |
| **Sección**   | _Completar con sección_      |
| **Fecha**     | Mayo 2026                    |

---

### 📌 Descripción del Problema

El objetivo de esta competencia es **predecir la década** en la que fue escrito un párrafo de texto.  
La década se representa como los **3 primeros dígitos del año** (ej: 1572 → 157).

Los textos son principalmente históricos en **español antiguo, latín e italiano**, con ruido de OCR.

### 🗺️ Estructura del Notebook

1. Instalación e importación de dependencias
2. Carga y exploración de datos (EDA)
3. Preprocesamiento y limpieza de texto
4. **Modelo 1**: MLP profundo con TF-IDF
5. **Modelo 2**: Red Neuronal Convolucional (CNN)
6. **Modelo 3**: LSTM Bidireccional (BiLSTM)
7. **Modelo 4**: Transfer Learning con mBERT
8. **Modelo 5**: Ensemble (votación de mejores modelos)
9. Comparación de modelos
10. Generación del archivo de entrega para Kaggle

---

## 1. 📦 Instalación de Dependencias

Instalamos todas las librerías necesarias para los modelos de aprendizaje profundo, procesamiento de texto y evaluación.  
Se usa `transformers` de HuggingFace para el modelo preentrenado mBERT.

In [5]:
import sys
import subprocess

print("=" * 70)
print("🔧 INSTALACIÓN Y VERIFICACIÓN DE DEPENDENCIAS")
print("=" * 70)
print(f"\n📍 Kernel Python: {sys.executable}")
print(f"📦 Entorno: {sys.prefix}\n")

# Lista de paquetes a instalar
PACKAGES = [
    'transformers',
    'datasets',
    'sentencepiece',
    'accelerate',
    'scikit-learn',
    'tensorflow',
    'torch',
    'torchvision',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'tqdm',
    'ipykernel'
]

print(f"📋 Instalando {len(PACKAGES)} paquetes...\n")

# Instalar cada paquete
for package in PACKAGES:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", package],
            check=True,
            timeout=300
        )
        print(f"✔ {package:<20} instalado")
    except Exception as e:
        print(f"✘ {package:<20} error: {str(e)[:40]}")

print("\n" + "=" * 70)
print("✅ VERIFICACIÓN DE IMPORTACIONES")
print("=" * 70 + "\n")

# Verificar que todos los paquetes se pueden importar
check_packages = {}
for package in PACKAGES:
    module_name = package.replace('-', '_')
    if package == 'scikit-learn':
        module_name = 'sklearn'
    elif package == 'pytorch':
        module_name = 'torch'
    
    try:
        __import__(module_name)
        # Obtener versión
        mod = __import__(module_name)
        version = getattr(mod, '__version__', 'desconocida')
        print(f"✔ {package:<20} v{version}")
        check_packages[package] = True
    except ImportError:
        print(f"✘ {package:<20} NO importable")
        check_packages[package] = False

print("\n" + "=" * 70)
total_ok = sum(check_packages.values())
print(f"✅ Resultado: {total_ok}/{len(PACKAGES)} paquetes disponibles")
print("=" * 70)

🔧 INSTALACIÓN Y VERIFICACIÓN DE DEPENDENCIAS

📍 Kernel Python: c:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\.venv\Scripts\python.exe
📦 Entorno: c:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\.venv

📋 Instalando 14 paquetes...

✔ transformers         instalado
✔ datasets             instalado
✔ sentencepiece        instalado
✔ accelerate           instalado
✔ scikit-learn         instalado
✔ tensorflow           instalado
✔ torch                instalado
✔ torchvision          instalado
✔ pandas               instalado
✔ numpy                instalado
✔ matplotlib           instalado
✔ seaborn              instalado
✔ tqdm                 instalado
✔ ipykernel            instalado

✅ VERIFICACIÓN DE IMPORTACIONES

✘ transformers         NO importable
✔ datasets             v4.8.5
✔ sentencepiece        v0.2.1
✔ accelerate           v1.13.0
✔ scikit-learn         v1.8.0

[TensorFlow DLL Diagnostic] Analyzing: c:\Us

## 2. 📚 Importación de Librerías

Importamos todas las librerías necesarias: TensorFlow/Keras para modelos secuenciales, HuggingFace Transformers para BERT, y scikit-learn para métricas y preprocesamiento.

In [6]:
# --- Librerías estándar ---
import os
import re
import json
import random
import warnings
warnings.filterwarnings('ignore')

# --- Manipulación de datos ---
import numpy as np
import pandas as pd

# --- Visualización ---
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')

# --- Scikit-learn ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier

# --- TensorFlow / Keras ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# --- HuggingFace Transformers ---
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset

# --- Reproducibilidad ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)

# --- Dispositivo GPU/CPU ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Librerías importadas correctamente.")
print(f"🖥️  TensorFlow versión: {tf.__version__}")
print(f"🔥 PyTorch versión: {torch.__version__}")
print(f"🤗 Dispositivo para PyTorch: {DEVICE}")
print(f"💡 GPUs disponibles (TF): {len(tf.config.list_physical_devices('GPU'))}")


[TensorFlow DLL Diagnostic] Analyzing: c:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\.venv\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd


ImportError: Traceback (most recent call last):
  File "c:\Users\User\Documents\UNI\Sexto Semestre\Machine Learning\Proyecto\Proyecto_2-ML\.venv\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: Module use of python312.dll conflicts with this version of Python.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

## 3. 📂 Carga y Exploración de Datos (EDA)

Cargamos los archivos `train.csv` y `eval.csv`, exploramos su estructura, distribución de clases y características del texto.  
Esto nos ayuda a entender el problema antes de diseñar los modelos.

> **Nota**: Asegúrate de que `train.csv` y `eval.csv` estén en el mismo directorio que este notebook, o ajusta el path.

In [ ]:
# ─── Carga de datos ───────────────────────────────────────────────
# Ajusta los paths si los archivos están en otra ubicación
TRAIN_PATH = 'train.csv'
EVAL_PATH  = 'eval.csv'

df_train = pd.read_csv(TRAIN_PATH)
df_eval  = pd.read_csv(EVAL_PATH)

print(f"📊 Tamaño del conjunto de entrenamiento : {df_train.shape}")
print(f"📊 Tamaño del conjunto de evaluación    : {df_eval.shape}")
print()
print("🔍 Primeras filas del conjunto de entrenamiento:")
df_train.head()

In [ ]:
# ─── Información básica ───────────────────────────────────────────
print("=== INFO TRAIN ===")
print(df_train.info())
print()
print("=== Valores nulos ===")
print(df_train.isnull().sum())
print()
print(f"🏷️  Número de décadas únicas : {df_train['decade'].nunique()}")
print(f"📅 Décadas min/max           : {df_train['decade'].min()} — {df_train['decade'].max()}")

In [ ]:
# ─── Distribución de décadas ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histograma de décadas
decade_counts = df_train['decade'].value_counts().sort_index()
axes[0].bar(decade_counts.index.astype(str), decade_counts.values, color='steelblue', alpha=0.8)
axes[0].set_title('Distribución de Décadas en Entrenamiento', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Década')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=90)

# Longitud de textos
df_train['text_len'] = df_train['text'].apply(lambda x: len(str(x).split()))
axes[1].hist(df_train['text_len'], bins=50, color='coral', alpha=0.8, edgecolor='black')
axes[1].set_title('Distribución de Longitud de Textos (palabras)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Número de palabras')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(df_train['text_len'].median(), color='navy', linestyle='--', label=f'Mediana={df_train["text_len"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_distribucion.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"📏 Estadísticas de longitud de texto:")
print(df_train['text_len'].describe())

In [ ]:
# ─── Muestra de textos por época ──────────────────────────────────
print("📖 Ejemplos de texto por época:\n")
for decade in sorted(df_train['decade'].unique())[:6]:
    sample = df_train[df_train['decade'] == decade]['text'].iloc[0]
    print(f"  Década {decade}xx: {str(sample)[:150]}...")
    print()

## 4. 🧹 Preprocesamiento y Limpieza de Texto

Los textos históricos presentan varios desafíos:
- **Ruido de OCR**: caracteres mal reconocidos (`ñ` → `ü`, espacios irregulares)
- **Ortografía antigua**: formas léxicas no estandarizadas
- **Mezcla de idiomas**: español, latín, italiano
- **Caracteres especiales** y puntuación ruidosa

La estrategia de limpieza debe ser **conservadora**: no eliminar demasiado, ya que la ortografía misma es una señal temporal importante.

In [ ]:
def clean_text(text: str, level: str = 'moderate') -> str:
    """
    Limpia el texto histórico.
    
    Niveles:
      - 'light'    : solo normalizar espacios y quitar caracteres de control
      - 'moderate' : light + eliminar puntuación excesiva y números aislados
      - 'heavy'    : moderate + lowercase + quitar stopwords muy cortas
    """
    if not isinstance(text, str):
        return ''
    
    # 1. Eliminar caracteres de control y tabulaciones
    text = re.sub(r'[\r\n\t]+', ' ', text)
    
    # 2. Normalizar guiones y rayas tipográficas
    text = re.sub(r'[—–\-]+', ' ', text)
    
    # 3. Eliminar secuencias de puntuación repetida
    text = re.sub(r'[.,:;!?¿¡]{2,}', '.', text)
    
    if level in ('moderate', 'heavy'):
        # 4. Eliminar números aislados (ruido de OCR de numeraciones)
        text = re.sub(r'\b\d{1,3}\b', '', text)
        
        # 5. Eliminar caracteres no alfanuméricos excepto puntuación básica
        text = re.sub(r'[^\w\s.,;:áéíóúàèìòùâêîôûãñüÁÉÍÓÚÀÈÌÒÙÂÊÎÔÛÃÑÜ]', ' ', text)
    
    if level == 'heavy':
        # 6. Minúsculas
        text = text.lower()
        
        # 7. Eliminar tokens de 1 carácter (ruido)
        text = re.sub(r'\b\w{1}\b', '', text)
    
    # 8. Normalizar espacios múltiples
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# Aplicar limpieza moderada (conserva señales temporales en ortografía)
df_train['text_clean'] = df_train['text'].apply(lambda x: clean_text(x, level='moderate'))
df_eval['text_clean']  = df_eval['text'].apply(lambda x: clean_text(x, level='moderate'))

# Eliminar filas donde el texto quedó vacío
df_train = df_train[df_train['text_clean'].str.len() > 5].reset_index(drop=True)

print(f"✅ Limpieza completada.")
print(f"   Filas train tras limpieza: {len(df_train)}")
print()
# Mostrar ejemplo de limpieza
idx = 0
print("🔤 ORIGINAL:")
print(repr(df_train['text'].iloc[idx][:200]))
print()
print("✨ LIMPIO:")
print(repr(df_train['text_clean'].iloc[idx][:200]))

## 5. 🏗️ Preparación de Etiquetas y División de Datos

Codificamos las décadas con `LabelEncoder` para convertirlas a índices enteros continuos (requerido por los modelos de clasificación). También dividimos en conjuntos de **entrenamiento (80%)** y **validación (20%)**.

In [ ]:
# ─── Codificación de etiquetas ────────────────────────────────────
label_encoder = LabelEncoder()
df_train['label'] = label_encoder.fit_transform(df_train['decade'])

NUM_CLASSES = len(label_encoder.classes_)
print(f"🏷️  Número de clases (décadas): {NUM_CLASSES}")
print(f"📅 Rango de décadas: {label_encoder.classes_.min()} — {label_encoder.classes_.max()}")

# Mapeo label → decade para recuperar predicciones
label_to_decade = dict(enumerate(label_encoder.classes_))
print(f"\n🗺️  Primeros 10 mapeos (label → decade): {dict(list(label_to_decade.items())[:10])}")

In [ ]:
# ─── División Train / Validación ──────────────────────────────────
X = df_train['text_clean'].values
y = df_train['label'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y  # mantener proporción de clases
)

X_test = df_eval['text_clean'].values

print(f"✅ División completada:")
print(f"   🔵 Train      : {len(X_train):>6} ejemplos")
print(f"   🟡 Validación : {len(X_val):>6} ejemplos")
print(f"   🔴 Test (eval): {len(X_test):>6} ejemplos")

## 6. 🤖 Modelo 1: MLP Profundo con TF-IDF

### Justificación
El primer modelo utiliza representación **TF-IDF** del texto combinada con una **red neuronal densa (MLP)** profunda.  
Es una buena línea de base profunda: rápida de entrenar y con buena capacidad de generalización sobre vocabulario histórico.

- **TF-IDF con n-gramas (1,2)**: captura palabras y bigramas frecuentes de cada época
- **MLP**: capas densas con BatchNormalization, Dropout y activaciones ReLU

In [ ]:
# ─── TF-IDF Vectorización ─────────────────────────────────────────
TFIDF_MAX_FEATURES = 50000

tfidf_vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 2),      # unigramas y bigramas
    sublinear_tf=True,       # suavizado logarítmico de TF
    min_df=2,                # ignorar términos que aparecen < 2 veces
    analyzer='char_wb',      # analizar a nivel de carácter (mejor para OCR)
    strip_accents=None       # conservar acentos (señal temporal)
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train).toarray().astype(np.float32)
X_val_tfidf   = tfidf_vectorizer.transform(X_val).toarray().astype(np.float32)
X_test_tfidf  = tfidf_vectorizer.transform(X_test).toarray().astype(np.float32)

print(f"✅ TF-IDF listo. Shape de X_train_tfidf: {X_train_tfidf.shape}")

In [ ]:
# ─── KNN como referencia rápida (sklearn) ─────────────────────────
# KNN con embeddings TF-IDF — útil para comparación
print("⏳ Entrenando KNN...")
knn = KNeighborsClassifier(n_neighbors=7, metric='cosine', n_jobs=-1)
knn.fit(X_train_tfidf, y_train)
knn_preds = knn.predict(X_val_tfidf)
knn_acc = accuracy_score(y_val, knn_preds)
print(f"🎯 KNN (k=7, cosine) Accuracy en validación: {knn_acc:.4f}")

In [ ]:
# ─── Construcción del modelo MLP ──────────────────────────────────
def build_mlp(input_dim: int, num_classes: int) -> keras.Model:
    """MLP profundo con BN, Dropout y skip connections suaves."""
    inputs = keras.Input(shape=(input_dim,), name='tfidf_input')
    
    x = layers.Dense(1024, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs, name='MLP_TF-IDF')
    return model


mlp_model = build_mlp(TFIDF_MAX_FEATURES, NUM_CLASSES)
mlp_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
mlp_model.summary()

In [ ]:
# ─── Entrenamiento MLP ────────────────────────────────────────────
callbacks_mlp = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_mlp.keras', monitor='val_accuracy', save_best_only=True, verbose=0)
]

print("⏳ Entrenando Modelo 1 — MLP con TF-IDF...")
history_mlp = mlp_model.fit(
    X_train_tfidf, y_train,
    validation_data=(X_val_tfidf, y_val),
    epochs=80,
    batch_size=256,
    callbacks=callbacks_mlp,
    verbose=1
)

mlp_val_acc = max(history_mlp.history['val_accuracy'])
print(f"\n✅ MLP — Mejor accuracy en validación: {mlp_val_acc:.4f}")

In [ ]:
# ─── Curva de aprendizaje MLP ─────────────────────────────────────
def plot_history(history, title='Modelo'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validación')
    axes[0].set_title(f'{title} — Accuracy')
    axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validación')
    axes[1].set_title(f'{title} — Loss')
    axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(f'history_{title.replace(" ","_")}.png', dpi=100)
    plt.show()

plot_history(history_mlp, 'MLP TF-IDF')

## 7. 🧠 Modelo 2: Red Neuronal Convolucional (CNN) para Texto

### Justificación
Las **CNN aplicadas a texto** son excelentes para capturar **patrones locales** como n-gramas y combinaciones de palabras.  
Utilizamos múltiples filtros de diferentes tamaños (3, 4, 5) para capturar contextos de distinto largo — muy útil para vocabulario histórico con variabilidad ortográfica.

- **Capa de Embedding**: aprende representaciones densas de tokens
- **Conv1D paralelas**: capturan trigramas, 4-gramas y 5-gramas
- **GlobalMaxPooling**: toma la característica más relevante de cada filtro

In [ ]:
# ─── Tokenización para modelos de secuencia ───────────────────────
MAX_VOCAB  = 60000
MAX_LEN    = 200   # truncar/rellenar a 200 tokens
EMBED_DIM  = 128

tokenizer_seq = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer_seq.fit_on_texts(X_train)

# Convertir textos a secuencias de enteros
X_train_seq = pad_sequences(tokenizer_seq.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post', truncating='post')
X_val_seq   = pad_sequences(tokenizer_seq.texts_to_sequences(X_val),   maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq  = pad_sequences(tokenizer_seq.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding='post', truncating='post')

VOCAB_SIZE = min(MAX_VOCAB, len(tokenizer_seq.word_index) + 1)
print(f"✅ Tokenización lista.")
print(f"   Vocabulario efectivo : {VOCAB_SIZE}")
print(f"   Shape X_train_seq    : {X_train_seq.shape}")

In [ ]:
# ─── Construcción del modelo CNN ──────────────────────────────────
def build_cnn(vocab_size: int, embed_dim: int, max_len: int, num_classes: int) -> keras.Model:
    """CNN multi-filtro para clasificación de texto."""
    inputs = keras.Input(shape=(max_len,), name='token_input')
    
    # Capa de embedding
    emb = layers.Embedding(vocab_size, embed_dim, mask_zero=False)(inputs)
    emb = layers.SpatialDropout1D(0.2)(emb)
    
    # Ramas paralelas con diferentes tamaños de kernel (n-gramas)
    branches = []
    for kernel_size in [3, 4, 5]:
        conv = layers.Conv1D(filters=256, kernel_size=kernel_size, activation='relu', padding='same')(emb)
        pool = layers.GlobalMaxPooling1D()(conv)
        branches.append(pool)
    
    # Concatenar ramas
    concat = layers.Concatenate()(branches)
    
    x = layers.Dense(256, activation='relu')(concat)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs, name='CNN_TextClassifier')
    return model


cnn_model = build_cnn(VOCAB_SIZE, EMBED_DIM, MAX_LEN, NUM_CLASSES)
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
cnn_model.summary()

In [ ]:
# ─── Entrenamiento CNN ────────────────────────────────────────────
callbacks_cnn = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_cnn.keras', monitor='val_accuracy', save_best_only=True, verbose=0)
]

print("⏳ Entrenando Modelo 2 — CNN para texto...")
history_cnn = cnn_model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=80,
    batch_size=128,
    callbacks=callbacks_cnn,
    verbose=1
)

cnn_val_acc = max(history_cnn.history['val_accuracy'])
print(f"\n✅ CNN — Mejor accuracy en validación: {cnn_val_acc:.4f}")
plot_history(history_cnn, 'CNN')

## 8. 🔄 Modelo 3: LSTM Bidireccional (BiLSTM)

### Justificación
Las **LSTM bidireccionales** capturan dependencias tanto hacia adelante como hacia atrás en el texto.  
Son especialmente útiles para textos históricos donde el contexto completo de la oración (no solo el orden izquierda-derecha) es informativo para determinar la época.

- **BiLSTM apiladas**: capturan jerarquías de contexto
- **Attention**: pondera qué partes del texto son más discriminativas temporalmente

In [ ]:
# ─── Capa de atención personalizada ──────────────────────────────
class AttentionLayer(layers.Layer):
    """Mecanismo de atención aditiva (Bahdanau) simplificado."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def build(self, input_shape):
        self.W = self.add_weight(name='att_weight', shape=(input_shape[-1], 1),
                                  initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='att_bias', shape=(input_shape[1], 1),
                                  initializer='zeros', trainable=True)
        super().build(input_shape)
    
    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)   # (batch, seq, 1)
        a = tf.nn.softmax(e, axis=1)                     # (batch, seq, 1)
        output = tf.reduce_sum(x * a, axis=1)            # (batch, hidden)
        return output


# ─── Construcción del modelo BiLSTM ───────────────────────────────
def build_bilstm(vocab_size: int, embed_dim: int, max_len: int, num_classes: int) -> keras.Model:
    """BiLSTM apiladas con mecanismo de atención."""
    inputs = keras.Input(shape=(max_len,), name='token_input')
    
    # Embedding con dropout espacial
    emb = layers.Embedding(vocab_size, embed_dim)(inputs)
    emb = layers.SpatialDropout1D(0.2)(emb)
    
    # Primera BiLSTM (retorna secuencias)
    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1)
    )(emb)
    
    # Segunda BiLSTM (retorna secuencias para atención)
    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.1)
    )(x)
    
    # Mecanismo de atención
    x = AttentionLayer(name='attention')(x)
    
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs, name='BiLSTM_Attention')
    return model


bilstm_model = build_bilstm(VOCAB_SIZE, EMBED_DIM, MAX_LEN, NUM_CLASSES)
bilstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
bilstm_model.summary()

In [ ]:
# ─── Entrenamiento BiLSTM ─────────────────────────────────────────
callbacks_bilstm = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_bilstm.keras', monitor='val_accuracy', save_best_only=True, verbose=0)
]

print("⏳ Entrenando Modelo 3 — BiLSTM con Atención...")
history_bilstm = bilstm_model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=80,
    batch_size=128,
    callbacks=callbacks_bilstm,
    verbose=1
)

bilstm_val_acc = max(history_bilstm.history['val_accuracy'])
print(f"\n✅ BiLSTM — Mejor accuracy en validación: {bilstm_val_acc:.4f}")
plot_history(history_bilstm, 'BiLSTM Atención')

## 9. 🤗 Modelo 4: Transfer Learning con mBERT

### Justificación
**BERT multilingüe (`bert-base-multilingual-cased`)** es un modelo preentrenado en 104 idiomas, incluyendo español y latín.  
El fine-tuning de este modelo es la aproximación más poderosa porque:
- Aprovecha conocimiento lingüístico preentrenado en textos históricos y modernos
- Las representaciones contextuales capturan matices semánticos y morfológicos de cada época
- El modelo `cased` preserva mayúsculas, importante para textos históricos

Usamos **DistilmBERT** (versión más ligera) para mayor velocidad si no hay GPU potente.

In [ ]:
# ─── Configuración del modelo preentrenado ────────────────────────
# Usar 'distilbert-base-multilingual-cased' para entornos sin GPU potente
# Usar 'bert-base-multilingual-cased' para mejor rendimiento (más lento)
MODEL_NAME = 'distilbert-base-multilingual-cased'
# MODEL_NAME = 'bert-base-multilingual-cased'  # <- descomenta para mBERT completo

BERT_MAX_LEN = 128  # máximo tokens por texto (BERT soporta hasta 512)

print(f"🤗 Cargando tokenizer de: {MODEL_NAME}")
bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ Tokenizer cargado.")

In [ ]:
# ─── Preparar dataset para HuggingFace Trainer ────────────────────
def tokenize_function(examples):
    """Tokeniza batch de textos con BERT tokenizer."""
    return bert_tokenizer(
        examples['text'],
        truncation=True,
        max_length=BERT_MAX_LEN,
        padding=False  # el DataCollator gestiona el padding dinámico
    )

# Crear DataFrames de train y validación
df_tr = pd.DataFrame({'text': X_train, 'label': y_train})
df_va = pd.DataFrame({'text': X_val,   'label': y_val})

# Convertir a HuggingFace Dataset
hf_train = Dataset.from_pandas(df_tr[['text','label']])
hf_val   = Dataset.from_pandas(df_va[['text','label']])

# Tokenizar
hf_train_tok = hf_train.map(tokenize_function, batched=True, remove_columns=['text'])
hf_val_tok   = hf_val.map(tokenize_function,   batched=True, remove_columns=['text'])

print(f"✅ Datasets tokenizados.")
print(f"   Train: {len(hf_train_tok)} | Val: {len(hf_val_tok)}")
print(f"   Columnas: {hf_train_tok.column_names}")

In [ ]:
# ─── Cargar modelo y configurar entrenamiento ─────────────────────
from transformers import AutoModelForSequenceClassification

print(f"🔄 Cargando modelo: {MODEL_NAME} con {NUM_CLASSES} clases de salida...")
bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)
bert_model = bert_model.to(DEVICE)
print("✅ Modelo cargado.")

# Función de métricas para el Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}


# Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir='./bert_checkpoints',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    save_total_limit=2,
    fp16=(DEVICE == 'cuda'),    # mixed precision solo si hay GPU
    report_to='none',
    seed=SEED,
    logging_steps=100,
)

data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=hf_train_tok,
    eval_dataset=hf_val_tok,
    tokenizer=bert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Trainer configurado.")

In [ ]:
# ─── Fine-tuning de mBERT ─────────────────────────────────────────
print(f"⏳ Iniciando fine-tuning de {MODEL_NAME}...")
print(f"   Esto puede tomar varios minutos (GPU recomendada).\n")

train_result = trainer.train()

print(f"\n✅ Fine-tuning completado.")
print(f"   Métricas de entrenamiento: {train_result.metrics}")

In [ ]:
# ─── Evaluación de mBERT en validación ───────────────────────────
bert_eval_results = trainer.evaluate()
bert_val_acc = bert_eval_results['eval_accuracy']
print(f"🎯 mBERT — Accuracy en validación: {bert_val_acc:.4f}")
print(f"   Resultados completos: {bert_eval_results}")

In [ ]:
# ─── Predicciones de mBERT en test ───────────────────────────────
# Tokenizar el conjunto de test
df_te = pd.DataFrame({'text': X_test, 'label': np.zeros(len(X_test), dtype=int)})
hf_test = Dataset.from_pandas(df_te[['text','label']])
hf_test_tok = hf_test.map(tokenize_function, batched=True, remove_columns=['text'])

# Predicciones
bert_preds_raw = trainer.predict(hf_test_tok)
bert_test_preds = np.argmax(bert_preds_raw.predictions, axis=-1)
print(f"✅ Predicciones mBERT en test generadas. Shape: {bert_test_preds.shape}")

# También guardar predicciones en validación para el ensemble
bert_val_preds_raw = trainer.predict(hf_val_tok)
bert_val_probs = tf.nn.softmax(bert_val_preds_raw.predictions, axis=-1).numpy()
print(f"✅ Probabilidades mBERT en validación guardadas.")

## 10. 🗳️ Modelo 5: Ensemble (Combinación de Modelos)

### Justificación
El **ensemble** combina las probabilidades de predicción de los mejores modelos para reducir la varianza y mejorar la precisión final.  
Utilizamos un **promedio ponderado de softmax** donde el peso de cada modelo es proporcional a su accuracy en validación.

Esta es una técnica estándar en competencias de ML que usualmente supera a cualquier modelo individual.

In [ ]:
# ─── Recopilar probabilidades de validación de cada modelo ────────
# MLP
mlp_val_probs  = mlp_model.predict(X_val_tfidf,  verbose=0)
# CNN
cnn_val_probs  = cnn_model.predict(X_val_seq,    verbose=0)
# BiLSTM
lstm_val_probs = bilstm_model.predict(X_val_seq, verbose=0)
# mBERT (ya calculado arriba como bert_val_probs)

# Accuracy individual en validación
models_acc = {
    'KNN'     : knn_acc,
    'MLP'     : accuracy_score(y_val, np.argmax(mlp_val_probs,  axis=-1)),
    'CNN'     : accuracy_score(y_val, np.argmax(cnn_val_probs,  axis=-1)),
    'BiLSTM'  : accuracy_score(y_val, np.argmax(lstm_val_probs, axis=-1)),
    'mBERT'   : bert_val_acc,
}

print("📊 Accuracy en validación por modelo:")
for name, acc in sorted(models_acc.items(), key=lambda x: -x[1]):
    bar = '█' * int(acc * 30)
    print(f"   {name:<10}: {acc:.4f}  {bar}")

In [ ]:
# ─── Ensemble ponderado por accuracy ──────────────────────────────
# Solo incluir en el ensemble modelos con accuracy > umbral
ENSEMBLE_THRESHOLD = 0.01  # incluir todos; ajustar si alguno es muy malo

# Construir pesos proporcionales al accuracy
ensemble_models = {
    'MLP'   : (mlp_val_probs,  models_acc['MLP']),
    'CNN'   : (cnn_val_probs,  models_acc['CNN']),
    'BiLSTM': (lstm_val_probs, models_acc['BiLSTM']),
    'mBERT' : (bert_val_probs, models_acc['mBERT']),
}

total_weight = sum(acc for _, acc in ensemble_models.values() if acc > ENSEMBLE_THRESHOLD)

# Promedio ponderado de probabilidades en validación
ensemble_val_probs = np.zeros_like(mlp_val_probs)
for name, (probs, acc) in ensemble_models.items():
    w = acc / total_weight
    ensemble_val_probs += w * probs
    print(f"   {name:<10}: peso = {w:.4f}")

ensemble_val_preds = np.argmax(ensemble_val_probs, axis=-1)
ensemble_acc = accuracy_score(y_val, ensemble_val_preds)
print(f"\n🏆 Ensemble ponderado — Accuracy en validación: {ensemble_acc:.4f}")

## 11. 📊 Comparación Final de Modelos

Visualizamos el rendimiento de todos los modelos en el conjunto de validación para seleccionar el mejor para la entrega en Kaggle.

In [ ]:
# ─── Tabla comparativa ────────────────────────────────────────────
all_results = {
    **models_acc,
    'Ensemble': ensemble_acc
}

results_df = pd.DataFrame(
    list(all_results.items()),
    columns=['Modelo', 'Accuracy Validación']
).sort_values('Accuracy Validación', ascending=False).reset_index(drop=True)

print("📊 RESUMEN COMPARATIVO DE MODELOS")
print("="*40)
print(results_df.to_string(index=False))
print()
print(f"🏆 Mejor modelo: {results_df.iloc[0]['Modelo']} ({results_df.iloc[0]['Accuracy Validación']:.4f})")

In [ ]:
# ─── Gráfico de barras comparativo ────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]
bars = ax.barh(
    results_df['Modelo'],
    results_df['Accuracy Validación'],
    color=colors, edgecolor='black', alpha=0.85
)
# Etiquetas de valor
for bar, val in zip(bars, results_df['Accuracy Validación']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold')

ax.set_xlim(0, results_df['Accuracy Validación'].max() + 0.05)
ax.set_xlabel('Accuracy en Validación', fontsize=12)
ax.set_title('Comparación de Modelos — Clasificación de Décadas', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('comparacion_modelos.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Matriz de confusión del mejor modelo ─────────────────────────
best_model_name = results_df.iloc[0]['Modelo']
print(f"🔍 Analizando el mejor modelo: {best_model_name}")

if best_model_name == 'Ensemble':
    best_val_preds = ensemble_val_preds
elif best_model_name == 'mBERT':
    best_val_preds = np.argmax(bert_val_probs, axis=-1)
elif best_model_name == 'CNN':
    best_val_preds = np.argmax(cnn_val_probs, axis=-1)
elif best_model_name == 'BiLSTM':
    best_val_preds = np.argmax(lstm_val_probs, axis=-1)
elif best_model_name == 'MLP':
    best_val_preds = np.argmax(mlp_val_probs, axis=-1)
else:
    best_val_preds = knn_preds

print(f"\n📋 Classification Report ({best_model_name}):")
print(classification_report(
    label_encoder.inverse_transform(y_val),
    label_encoder.inverse_transform(best_val_preds),
    zero_division=0
))

## 12. 📤 Generación del Archivo de Entrega para Kaggle

Generamos las predicciones sobre el conjunto de evaluación (`eval.csv`) usando el **mejor modelo** identificado en la sección anterior.  
El archivo de salida tiene el formato requerido por Kaggle: `id,answer`.

In [ ]:
# ─── Predicciones del mejor modelo en TEST ────────────────────────
print(f"🔮 Generando predicciones con: {best_model_name}")

if best_model_name == 'Ensemble':
    # Predicciones de cada modelo en test
    mlp_test_probs  = mlp_model.predict(X_test_tfidf, verbose=0)
    cnn_test_probs  = cnn_model.predict(X_test_seq,   verbose=0)
    lstm_test_probs = bilstm_model.predict(X_test_seq, verbose=0)
    # bert_test_preds ya fue calculado arriba, necesitamos probs
    bert_test_probs = tf.nn.softmax(bert_preds_raw.predictions, axis=-1).numpy()
    
    # Ensemble ponderado en test
    final_probs = np.zeros_like(mlp_test_probs)
    for name, (val_probs, acc) in ensemble_models.items():
        w = acc / total_weight
        if name == 'MLP':    final_probs += w * mlp_test_probs
        elif name == 'CNN':  final_probs += w * cnn_test_probs
        elif name == 'BiLSTM': final_probs += w * lstm_test_probs
        elif name == 'mBERT': final_probs += w * bert_test_probs
    
    test_label_preds = np.argmax(final_probs, axis=-1)

elif best_model_name == 'mBERT':
    test_label_preds = bert_test_preds

elif best_model_name == 'CNN':
    test_label_preds = np.argmax(cnn_model.predict(X_test_seq, verbose=0), axis=-1)

elif best_model_name == 'BiLSTM':
    test_label_preds = np.argmax(bilstm_model.predict(X_test_seq, verbose=0), axis=-1)

elif best_model_name == 'MLP':
    test_label_preds = np.argmax(mlp_model.predict(X_test_tfidf, verbose=0), axis=-1)

else:  # KNN
    test_label_preds = knn.predict(X_test_tfidf)

# Decodificar etiquetas a décadas reales
test_decade_preds = label_encoder.inverse_transform(test_label_preds)

print(f"✅ {len(test_decade_preds)} predicciones generadas.")
print(f"   Distribución de décadas predichas:")
unique, counts = np.unique(test_decade_preds, return_counts=True)
for d, c in zip(unique[:10], counts[:10]):
    print(f"     Década {d}: {c} predicciones")

In [ ]:
# ─── Crear y guardar el archivo de entrega ────────────────────────
submission_df = pd.DataFrame({
    'id'    : df_eval['id'],
    'answer': test_decade_preds
})

SUBMISSION_FILE = f'submission_{best_model_name.lower().replace(" ","_")}.csv'
submission_df.to_csv(SUBMISSION_FILE, index=False)

print(f"✅ Archivo de entrega generado: '{SUBMISSION_FILE}'")
print(f"   Filas en el archivo  : {len(submission_df)}")
print()
print("📄 Primeras 10 filas:")
print(submission_df.head(10).to_string(index=False))

In [ ]:
# ─── Verificación del formato del archivo ─────────────────────────
check = pd.read_csv(SUBMISSION_FILE)

assert 'id' in check.columns,     "❌ Falta columna 'id'"
assert 'answer' in check.columns, "❌ Falta columna 'answer'"
assert len(check) == len(df_eval), f"❌ Número de filas incorrecto: {len(check)} vs {len(df_eval)}"
assert check['id'].equals(df_eval['id']), "❌ IDs no coinciden"

print("✅ Verificación del archivo de entrega:")
print(f"   ✔ Columnas correctas : {list(check.columns)}")
print(f"   ✔ Número de filas    : {len(check)}")
print(f"   ✔ IDs verificados    : OK")
print(f"   ✔ Archivo listo para subir a Kaggle: '{SUBMISSION_FILE}'")

## 13. 💾 Guardado de Modelos y Artefactos

Guardamos todos los modelos y artefactos necesarios para reproducibilidad y para la entrega en Bloque Neon.

In [ ]:
import pickle

# ─── Guardar modelos Keras ─────────────────────────────────────────
mlp_model.save('modelo_mlp_final.keras')
cnn_model.save('modelo_cnn_final.keras')
bilstm_model.save('modelo_bilstm_final.keras')
print("✅ Modelos Keras guardados.")

# ─── Guardar modelo mBERT ─────────────────────────────────────────
bert_model.save_pretrained('./modelo_mbert_final')
bert_tokenizer.save_pretrained('./modelo_mbert_final')
print("✅ mBERT guardado en './modelo_mbert_final'.")

# ─── Guardar preprocesadores ──────────────────────────────────────
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

with open('tokenizer_seq.pkl', 'wb') as f:
    pickle.dump(tokenizer_seq, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print("✅ Preprocesadores guardados.")

# ─── Guardar KNN ──────────────────────────────────────────────────
with open('modelo_knn.pkl', 'wb') as f:
    pickle.dump(knn, f)

print("✅ KNN guardado.")

# ─── Guardar métricas finales ─────────────────────────────────────
metrics_summary = {
    'best_model'          : best_model_name,
    'val_accuracy_by_model': all_results,
    'submission_file'     : SUBMISSION_FILE
}

with open('metrics_summary.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2, default=float)

print("\n📋 Resumen guardado en 'metrics_summary.json'")
print(json.dumps(metrics_summary, indent=2, default=float))

## 14. 📝 Conclusiones

### Resumen del proceso

En esta etapa del proyecto implementamos y comparamos **5 aproximaciones** de aprendizaje profundo para la clasificación de textos históricos por década:

| Modelo | Descripción | Fortaleza |
|--------|-------------|------------|
| **KNN + TF-IDF** | Vecinos más cercanos sobre representación TF-IDF | Línea de base simple |
| **MLP + TF-IDF** | Red densa profunda sobre características TF-IDF | Rápido y efectivo con vocabulario fijo |
| **CNN** | Filtros convolucionales multi-escala sobre embeddings | Captura patrones locales de n-gramas |
| **BiLSTM + Atención** | LSTM bidireccional con mecanismo de atención | Captura dependencias largas y contexto |
| **mBERT (fine-tuned)** | Transferencia desde BERT multilingüe | Mejor generalización y contexto semántico |
| **Ensemble** | Promedio ponderado de los 4 modelos DL | Reduce varianza, mejor generalización |

### Decisiones clave
- Se usó limpieza **moderada** del texto para preservar señales ortográficas históricas
- Se usó `DistilmBERT` multilingüe para manejar español, latín e italiano
- El ensemble pondera los modelos por su accuracy en validación

### Posibles mejoras
- Aumentación de datos con fuentes externas de textos históricos (Project Gutenberg, etc.)
- Ajuste de hiperparámetros con búsqueda bayesiana
- Usar modelos más específicos para español histórico (e.g., `PlanTL-GOB-ES/roberta-base-bne`)
- Datos sintéticos generados con LLMs prompting sobre épocas específicas

---
*Notebook generado para la Etapa 2 — Aprendizaje de Máquina 2026-10, Universidad de los Andes.*